Multi-Agent Software Bug Diagnosis System

In [ ]:
#cell -1

# Install required libraries
!pip install -q transformers accelerate sentencepiece

In [ ]:
#cell -2

from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
#cell - 3

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2" #only if GPU available
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512
)

print("✅ Model Loaded")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Model Loaded


In [ ]:
#cell -4

def ask_llm(prompt):
    output = llm(
        prompt,
        do_sample=False,
        max_new_tokens=300
    )[0]["generated_text"]

    # 🔥 REMOVE prompt echo
    return output.replace(prompt, "").strip()

In [ ]:
#cell -5

# ===== INPUT CODE =====
source_code = """
x = "10"
y = 5
print(x + y)
"""
source_code
# ===== ERROR LOG =====
error_log = """
TypeError: can only concatenate str (not "int") to str
"""

In [ ]:
#cell -6

def log_analyzer(error_log):
    prompt = f"""[INST]
You are a Log Analyzer Agent.

Return only:
- Error type
- Root cause
- Location

ERROR LOG:
{error_log}
[/INST]"""
    return ask_llm(prompt) #calling the llm model to generate the output

In [ ]:
#cell -7

# Agent who will read the code:
def code_reader(source_code):
    prompt = f"""[INST]
You are a Code Reader Agent.
Rules:
- Be strict
Return ONLY:
Purpose:
Risky lines:
Suspicious patterns:

CODE:
{source_code}
[/INST]"""
    return ask_llm(prompt)

In [ ]:
#cell -8

#agent who will debug the code:
def debugger_agent(code, log_analysis, code_analysis):
    prompt = f"""[INST]
You are a Debugger Agent.
STRICT RULES:
- Do NOT remove functionality
- Do NOT return empty functions
- You MUST handle explicitly
Respond EXACTLY in this format (only if you find any BUG):
BUG:
FIXED CODE:
EXPLANATION:

ORIGINAL CODE:
{code}
[/INST]"""
    return ask_llm(prompt)

In [ ]:
#cell -9

#agent who will review debuger agent

def reviewer_agent(debugger_agent):
    prompt = f"""[INST]
You are a Reviewer Agent.

Answer ONLY:
- Is fix correct? (Yes/No)
- Any missing edge cases? (If none, say None)
- Final recommendation (1 line)

DEBUGGER OUTPUT:
{debugger_agent}
[/INST]"""
    return ask_llm(prompt)

In [ ]:
#cell -10

# Run agents (make sure this cell is executed)

log_analysis = log_analyzer(error_log)   # or log_analyzer(error_log)
code_analysis = code_reader(source_code) # or code_reader(source_code)
debugger_output = debugger_agent(
    source_code,
    log_analysis,
    code_analysis
)
review_output = reviewer_agent(debugger_output)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/

In [ ]:
#cell -11

print("🧠 FINAL BUG DIAGNOSIS REPORT")
print("="*50)
print("🔍 Error Analysis:\n", log_analysis)
print("="*50)
print("\n📖 Code Understanding:\n", code_analysis)
print("="*50)
print("\n🐞 Proposed Fix:\n", debugger_output)
print("="*50)
print("\n✅ Review Verdict:\n", review_output)

🧠 FINAL BUG DIAGNOSIS REPORT
🔍 Error Analysis:
 Error type: TypeError
Root cause: Attempting to concatenate an int to a str without using proper conversion methods.
Location: The exact location of this error would depend on the specific code causing the issue. However, the error message is typically found in a log file generated by the application server or framework.

📖 Code Understanding:
 Purpose: The purpose of this code is to add the value of an integer (y) to a string (x) and print the resulting concatenated string.

Risky lines: The risky line in this code is the addition of a string (x) and an integer (y) without using the appropriate conversion functions. This can lead to unexpected results or errors.

Suspicious patterns: There is no suspicious pattern in this code as it is a simple and straightforward statement. However, the potential risk of adding a string and an integer without proper conversion should be noted.

🐞 Proposed Fix:
 ORIGINAL CODE:

x = "10"
y = 5
print(x + y